#1- Extracción desde Capa Bronze

In [0]:
df_bronze = spark.table("workspace.tp_dnrpa_bronze.bronze_transferencias")

## Importamos librerias


In [0]:

# Módulo principal para aplicar transformaciones a las columnas (casteos, fechas, lógicas)
from pyspark.sql import functions as F

# Módulo para ejecutar funciones de ventana (necesario para la deduplicación)
from pyspark.sql.window import Window

# Módulo de tipos de datos (opcional pero recomendado para tipado estricto)
from pyspark.sql.types import IntegerType, DateType, StringType

#2- Limpieza de datos

In [0]:
#Eliminamos columnas que no son relevantes
cols_to_exclude = ["_rescued_data", "titular_pais_nacimiento_id", "titular_genero", "titular_anio_nacimiento"]
df_silver = df_bronze.select([col for col in df_bronze.columns if col not in cols_to_exclude])

In [0]:
# Creamos columna de tramite
df_silver = df_silver.withColumn("tramite",F.when(F.col("tramite_tipo").rlike("(?i)inscripcion|inscripción"), "ALTA").
                                 when(F.col("tramite_tipo").rlike("(?i)baja"), "BAJA").
                                 when(F.col("tramite_tipo").rlike("(?i)transferencia"), "TRANSFERENCIA").otherwise(None))



In [0]:
# Filtramos solo las transferencias usando la columna 'tramite' creada anteriormente
df_silver = df_silver.filter(F.col("tramite") == "TRANSFERENCIA")


In [0]:
df_silver.display(limit=10)

##Limpieza de Marca y Marca_ID


In [0]:
# Normalizamos los nombres de las marcas: para cada código, usar el nombre más común
df_silver.createOrReplaceTempView("df_silver_temp")

df_silver = spark.sql("""
  WITH marca_normalizada AS (
    SELECT 
      automotor_marca_codigo,
      automotor_marca_descripcion,
      COUNT(*) as conteo,
      ROW_NUMBER() OVER (PARTITION BY automotor_marca_codigo ORDER BY COUNT(*) DESC) as rank
    FROM df_silver_temp
    WHERE automotor_marca_codigo IS NOT NULL
    GROUP BY automotor_marca_codigo, automotor_marca_descripcion
  )
  SELECT 
    t.tramite_tipo,
    t.tramite_fecha,
    t.fecha_inscripcion_inicial,
    t.registro_seccional_codigo,
    t.registro_seccional_descripcion,
    t.registro_seccional_provincia,
    t.automotor_origen,
    t.automotor_anio_modelo,
    t.automotor_tipo_codigo,
    t.automotor_tipo_descripcion,
    t.automotor_marca_codigo,
    COALESCE(m.automotor_marca_descripcion, t.automotor_marca_descripcion) as automotor_marca_descripcion,
    t.automotor_modelo_codigo,
    t.automotor_modelo_descripcion,
    t.automotor_uso_codigo,
    t.automotor_uso_descripcion,
    t.titular_tipo_persona,
    t.titular_domicilio_localidad,
    t.titular_domicilio_provincia,
    t.titular_pais_nacimiento,
    t.titular_porcentaje_titularidad,
    t.titular_domicilio_provincia_id,
    t.tramite
  FROM df_silver_temp t
  LEFT JOIN marca_normalizada m 
    ON t.automotor_marca_codigo = m.automotor_marca_codigo 
    AND m.rank = 1
""")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Limpieza base para TODAS las filas (caracteres raros y espacios extra)
df_silver = df_silver.withColumn(
    "marca_temp",
    F.regexp_replace(F.trim(F.upper(F.col("automotor_marca_descripcion"))), r'[^A-Z0-9 ]', '')
)
# Reemplazamos múltiples espacios internos por un solo espacio
df_silver = df_silver.withColumn(
    "marca_temp",
    F.regexp_replace(F.col("marca_temp"), r'\s+', ' ')
)

# 2. Diccionario de marcas principales y COMPUESTAS (vital para que "ALFA ROMEO" no se corte a "ALFA")
mapeo_principal = {
    "VOLKSWAGEN": "VOLK|VOLS|VW|WOLK|VOKS",
    "CHEVROLET": "CHEVROLET|CHEVR|CHEV|CHEROLET|VHEVROLET|CHVROLET|CHRVROLET",
    "RENAULT": "RENAULT|RENAU",
    "FORD": "FORD",
    "FIAT": "FIAT",
    "PEUGEOT": "PEUGEOT|PEUG",
    "TOYOTA": "TOYOTA|YOYOTA",
    "MERCEDES BENZ": "MERCEDES BENZ|MERCEDESBENZ|MERCEDES|MERCEDEZ|MBENZ|MERCDES|MRECEDES|MERCCEDES|MERCEDS",
    "CITROEN": "CITROEN|CITRO",
    "HONDA": "HONDA",
    "NISSAN": "NISSAN",
    "SUZUKI": "SUZUKI|SUSUKI",
    "DODGE": "DODGE",
    "JEEP": "JEEP",
    "IKA": "IKA",
    "RAM": "RAM",
    "MITSUBISHI": "MITSUBISHI|MITSUBUSHI|MITSUBISH",
    "HYUNDAI": "HYUNDAI",
    "KIA": "KIA",
    "AUDI": "AUDI",
    "BMW": r"BMW|B M W|B\.M\.W",
    "CHRYSLER": "CHRYSLER",
    "SCANIA": "SCANIA",
    "VOLVO": "VOLVO",
    "IVECO": "IVECO",
    "ISUZU": "ISUZU|IZUZU",
    "LAND ROVER": "LAND ROVER|LANDROVER",
    "ROVER": "ROVER",
    "ALFA ROMEO": "ALFA ROMEO|ALFA",
    "DAIHATSU": "DAIHATSU|DAITHASU|DAHIATSU",
    "PORSCHE": "PORSCHE|PORCHE",
    "SEAT": "SEAT",
    "MAZDA": "MAZDA",
    "LADA": "LADA",
    "CHERY": "CHERY",
    "SSANGYONG": "SSANGYONG",
    "SUBARU": "SUBARU",
    "JAGUAR": "JAGUAR",
    "SAAB": "SAAB",
    "OPEL": "OPEL",
    "RASTROJERO": "RASTROJERO|RESTROJERO|RSATROJERO",
    "TORINO": "TORINO",
    "RAMBLER": "RAMBLER",
    "AUTO UNION": "AUTO UNION|DKW",
    "SIAM DI TELLA": "SIAM|DI TELLA",
    "BORGWARD": "BORGWARD",
    "DEUTZ": "DEUTZ",
    "BEDFORD": "BEDFORD",
    "DAEWOO": "DAEWOO|DAEWO",
    "AUSTIN": "AUSTIN",
    "WILLYS": "WILLYS|WYLLYS|WYLLIS|WILLIS|WILLY",
    "VALIANT": "VALIANT"
}

# 3. Aplicamos el diccionario
condicion_marca = F.col("marca_temp")
for marca_oficial, variaciones in mapeo_principal.items():
    condicion_marca = F.when(
        F.col("marca_temp").rlike(variaciones), marca_oficial
    ).otherwise(condicion_marca)

df_silver = df_silver.withColumn("marca_temp", condicion_marca)

# 4. NORMALIZACIÓN DINÁMICA PARA EL RESTO (La "Cola Larga")
marcas_oficiales = list(mapeo_principal.keys())

df_silver = df_silver.withColumn(
    "automotor_marca_descripcion",
    F.when(
        F.col("marca_temp").isin(marcas_oficiales), 
        F.col("marca_temp") # Si ya está arreglada, la dejamos intacta
    ).otherwise(
        # Si es una marca rara que no está en el dict, extraemos solo su primera palabra
        F.split(F.col("marca_temp"), " ").getItem(0)
    )
).drop("marca_temp")

# 5. Calcular y asignar el ID Maestro para TODAS las marcas normalizadas
df_codigos_validos = df_silver.filter(
    F.col("automotor_marca_codigo").isNotNull() & 
    (F.lower(F.col("automotor_marca_codigo")) != 'null') & 
    (F.col("automotor_marca_codigo") != '')
).groupBy("automotor_marca_descripcion", "automotor_marca_codigo").agg(F.count("*").alias("conteo"))

window_spec = Window.partitionBy("automotor_marca_descripcion").orderBy(F.col("conteo").desc())

df_maestro = (
    df_codigos_validos.withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .select("automotor_marca_descripcion", F.col("automotor_marca_codigo").alias("codigo_nuevo"))
)

# 6. Cruzar y actualizar
df_silver = df_silver.join(df_maestro, on="automotor_marca_descripcion", how="left")

df_silver = (
    df_silver.withColumn(
        "automotor_marca_codigo", 
        F.coalesce(F.col("codigo_nuevo"), F.col("automotor_marca_codigo"))
    )
    .drop("codigo_nuevo")
)

###Reemplazo de null y creación de ID

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Identificar las marcas que actualmente no tienen ID
df_sin_id = df_silver.filter(
    F.col("automotor_marca_codigo").isNull() | 
    (F.lower(F.col("automotor_marca_codigo")) == 'null') | 
    (F.col("automotor_marca_codigo") == '')
)

# 2. Agrupar, contar y filtrar solo las que tienen más de 10 registros (DESTACADAS)
marcas_destacadas = (
    df_sin_id.groupBy("automotor_marca_descripcion")
    .agg(F.count("*").alias("conteo"))
    .filter(F.col("conteo") > 10)
)

# 3. Generar un ID único sintético para estas marcas destacadas (comenzando en 9001)
window_spec = Window.orderBy(F.col("conteo").desc())

df_nuevos_ids = marcas_destacadas.withColumn(
    "nuevo_codigo", 
    (F.row_number().over(window_spec) + 9000).cast("string")
).select("automotor_marca_descripcion", "nuevo_codigo")

# 4. Cruzar con el DataFrame original (df_silver)
df_silver = df_silver.join(df_nuevos_ids, on="automotor_marca_descripcion", how="left")

# 5. Actualizar la columna original solo donde aplique el nuevo código
df_silver = (
    df_silver.withColumn(
        "automotor_marca_codigo", 
        F.coalesce(F.col("automotor_marca_codigo"), F.col("nuevo_codigo"))
    )
    .drop("nuevo_codigo")
)

# 6. PARA MARCAS RARAS (<=10 registros): asignar IDs únicos desde 9999 en adelante
# 6.1. Identificar marcas que aún no tienen código
df_marcas_raras = df_silver.filter(
    F.col("automotor_marca_codigo").isNull() | 
    (F.lower(F.col("automotor_marca_codigo")) == 'null') | 
    (F.col("automotor_marca_codigo") == '')
).select("automotor_marca_descripcion").distinct()

# 6.2. Asignar IDs únicos empezando desde 9999
window_spec_raras = Window.orderBy("automotor_marca_descripcion")

df_ids_raras = df_marcas_raras.withColumn(
    "codigo_raro",
    (F.row_number().over(window_spec_raras) + 9998).cast("string")
)

# 6.3. Cruzar y actualizar
df_silver = df_silver.join(df_ids_raras, on="automotor_marca_descripcion", how="left")

df_silver = (
    df_silver.withColumn(
        "automotor_marca_codigo",
        F.coalesce(F.col("automotor_marca_codigo"), F.col("codigo_raro"))
    )
    .drop("codigo_raro")
)

print("✓ Limpieza de marcas completada con IDs únicos para cada marca")

##Limpieza por Tipo y Tipo_ID

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# PASO 1: Normalización SQL por Frecuencia (Similar a Celda 10)
# ============================================================
df_silver.createOrReplaceTempView("df_silver_tipo_temp")

df_silver = spark.sql("""
  WITH tipo_normalizado AS (
    SELECT 
      automotor_tipo_codigo,
      automotor_tipo_descripcion,
      COUNT(*) as conteo,
      ROW_NUMBER() OVER (PARTITION BY automotor_tipo_codigo ORDER BY COUNT(*) DESC) as rank
    FROM df_silver_tipo_temp
    WHERE automotor_tipo_codigo IS NOT NULL
    GROUP BY automotor_tipo_codigo, automotor_tipo_descripcion
  )
  SELECT 
    t.tramite_tipo,
    t.tramite_fecha,
    t.fecha_inscripcion_inicial,
    t.registro_seccional_codigo,
    t.registro_seccional_descripcion,
    t.registro_seccional_provincia,
    t.automotor_origen,
    t.automotor_anio_modelo,
    t.automotor_tipo_codigo,
    COALESCE(tn.automotor_tipo_descripcion, t.automotor_tipo_descripcion) as automotor_tipo_descripcion,
    t.automotor_marca_codigo,
    t.automotor_marca_descripcion,
    t.automotor_modelo_codigo,
    t.automotor_modelo_descripcion,
    t.automotor_uso_codigo,
    t.automotor_uso_descripcion,
    t.titular_tipo_persona,
    t.titular_domicilio_localidad,
    t.titular_domicilio_provincia,
    t.titular_pais_nacimiento,
    t.titular_porcentaje_titularidad,
    t.titular_domicilio_provincia_id,
    t.tramite
  FROM df_silver_tipo_temp t
  LEFT JOIN tipo_normalizado tn 
    ON t.automotor_tipo_codigo = tn.automotor_tipo_codigo 
    AND tn.rank = 1
""")

# ============================================================
# PASO 2: Normalización PySpark con Diccionario (Similar a Celda 11)
# ============================================================

# 2.1. Limpieza base para TODAS las filas (caracteres raros y espacios extra)
df_silver = df_silver.withColumn(
    "tipo_temp",
    F.regexp_replace(F.trim(F.upper(F.col("automotor_tipo_descripcion"))), r'[^A-Z0-9 /]', '')
)
# Reemplazamos múltiples espacios internos por un solo espacio
df_silver = df_silver.withColumn(
    "tipo_temp",
    F.regexp_replace(F.col("tipo_temp"), r'\s+', ' ')
)

# 2.2. Diccionario de tipos principales de vehículos
mapeo_tipo = {
    "SEDAN 5 PUERTAS": "SEDAN 5 PUERTAS|SEDAN 5 PTAS|SEDAN 5 P",
    "SEDAN 4 PUERTAS": "SEDAN 4 PUERTAS|SEDAN 4 PTAS|SEDAN 4 P|BERLINA 4 PUERTAS|BERLINA 4 PTAS",
    "SEDAN 3 PUERTAS": "SEDAN 3 PUERTAS|SEDAN 3 PTAS|SEDAN 3 P|BERLINA 3 PUERTAS|BERLINA 3 PTAS| 3 PTAS",
    "SEDAN 2 PUERTAS": "SEDAN 2 PUERTAS|SEDAN 2 PTAS|SEDAN 2 P",
    "SEDAN": "^SEDAN$|^BERLINA$",
    "PICK-UP": "PICK UP|PICKUP",
    "PICK-UP CABINA DOBLE": "PICK UP CABINA DOBLE|PICKUP CABINA DOBLE",
    "PICK-UP CABINA SIMPLE": "PICK UP CABINA SIMPLE|PICKUP CABINA SIMPLE|PICK UP CABINA SIMPL",
    "PICK-UP CABINA Y MEDIA": "PICK UP CABINA Y MEDIA|PICKUP CABINA Y MEDIA|PICK UP CABINA Y MED",
    "RURAL 5 PUERTAS": "RURAL 5 PUERTAS|RURAL 5 PTAS|RURAL 4/5 PUERTAS|RURAL 4/5 PTAS",
    "RURAL 4 PUERTAS": "RURAL 4 PUERTAS|RURAL 4 PTAS",
    "RURAL 3 PUERTAS": "RURAL 3 PUERTAS|RURAL 3 PTAS|RURAL 2/3 PTAS",
    "RURAL": "^RURAL$",
    "FURGON": "^FURGON$|FURGON 600|FURGON 800|FURGON 3000|FURGON 3550|FURGON LARGO",
    "FURGON VIDRIADO C/ASIENTOS": "FURGON VIDRIADO C/ASIENTOS|FURGON VIDRIADO CON ASIENTOS|FURGON VID C/ASIENTOS|FURGON VID C/ ASIENTOS|FURGON VIDRIADO C/AS|FURGON VID C ASIENTOS|FURGON VIDRIADO C ASIENTOS|FOURGON COURT TYPE 600|FURGON COURT TYPE 600",
    "FURGON VIDRIADO": "FURGON VIDRIADO$|FURGON VID$",
    "FURGONETA": "FURGONETA|FURGONETA VIDRIADA",
    "TODO TERRENO": "TODO TERRENO",
    "CHASIS C/CABINA": "CHASIS C/CABINA|CHASIS CON CABINA|CHASSIS C/CABINA|CHASSIS CON CABINA|CHASIS C/ CABINA|CHASIS C/CABINA P/CAMION|CHASIS C/CABINA P/CA",
    "CHASIS S/CABINA": "CHASIS S/CABINA|CHASIS SIN CABINA",
    "CHASIS C/CABINA DORMITORIO": "CHASIS C/CABINA DORMITORIO|CHASIS CON CABINA DORMITORIO|CHASIS C/ CABINA DORMITORIO|CHASIS C/ CABINA DOR",
    "COUPE": "^COUPE$|COUPE 2 PUERTAS|COUPE 2 PTAS|COUPE/MICROCOUPE",
    "CONVERTIBLE": "CONVERTIBLE|DESCAPOTABLE",
    "FAMILIAR": "FAMILIAR|FAMILIAR 5 ASIENTOS|STATION WAGON|BREAK",
    "UTILITARIO": "UTILITARIO",
    "CAMION": "^CAMION$|CAMION TRACTOR",
    "CAMION GRUA": "CAMION GRUA",
    "CAMION HORMIGONERO": "CAMION HORMIGONERO",
    "CAMION AUTOBOMBA": "CAMION AUTOBOMBA",
    "CAMION C/CABINA DORMITORIO": "CAMION C/CABINA DORMITORIO",
    "TRACTOR DE CARRETERA": "TRACTOR DE CARRETERA|TRACTOR C/CABINA DORMITORIO|TRACTOR C/ CABINA DORMITORIO|TRACTOR C/CABINA DOR|UNIDAD TRACTORA",
    "SEMIRREMOLQUE": "SEMIRREMOLQUE|SEMIACOPLADO|SEMI ACOPLADO|SEMIRREMOLQUE TANQUE|SEMIRREMOLQUE BATEA|SEMIRREMOLQUE BITREN",
    "ACOPLADO": "^ACOPLADO$|ACOPLADO PLAYO",
    "TRAILER": "TRAILER|REMOLQUE|REMOLQUE GASTRONOMICO",
    "OMNIBUS": "OMNIBUS",
    "MINIBUS": "MINIBUS|MICROOMNIBUS|MIDIBUS",
    "TRANSPORTE DE PASAJEROS": "TRANSPORTE DE PASAJEROS|TRANSP DE PASAJEROS|TRANS DE PASAJEROS",
    "TRANSPORTE DE CARGA": "TRANSPORTE DE CARGA|TRANSP DE CARGA|TRANSPORTE UTILITARIO",
    "CASA RODANTE": "CASA RODANTE|MOTORHOME|CASA RODANTE C/MOTOR|CASA RODANTE CON MOTOR|CASA RODANTE MOTORIZADA|CASA RODANTE AUTOPROPULSADA",
    "AMBULANCIA": "AMBULANCIA",
    "ARENERO": "ARENERO",
    "CARRETON": "CARRETON",
    "CUATRICICLO": "CUADRICICLO|CUATRICICLO"
}

# 2.3. Aplicamos el diccionario
condicion_tipo = F.col("tipo_temp")
for tipo_oficial, variaciones in mapeo_tipo.items():
    condicion_tipo = F.when(
        F.col("tipo_temp").rlike(variaciones), tipo_oficial
    ).otherwise(condicion_tipo)

df_silver = df_silver.withColumn("tipo_temp", condicion_tipo)

# 2.4. NORMALIZACIÓN DINÁMICA PARA EL RESTO (solo primera palabra para evitar errores)
tipos_oficiales = list(mapeo_tipo.keys())

df_silver = df_silver.withColumn(
    "automotor_tipo_descripcion",
    F.when(
        F.col("tipo_temp").isin(tipos_oficiales), 
        F.col("tipo_temp")
    ).otherwise(
        # Extraer solo la primera palabra para tipos no contemplados
        F.split(F.col("tipo_temp"), " ").getItem(0)
    )
).drop("tipo_temp")

# 2.5. Calcular y asignar el ID Maestro para TODOS los tipos normalizados
df_codigos_tipo_validos = df_silver.filter(
    F.col("automotor_tipo_codigo").isNotNull() & 
    (F.lower(F.col("automotor_tipo_codigo")) != 'null') & 
    (F.col("automotor_tipo_codigo") != '')
).groupBy("automotor_tipo_descripcion", "automotor_tipo_codigo").agg(F.count("*").alias("conteo"))

window_spec_tipo = Window.partitionBy("automotor_tipo_descripcion").orderBy(F.col("conteo").desc())

df_maestro_tipo = (
    df_codigos_tipo_validos.withColumn("rn", F.row_number().over(window_spec_tipo))
    .filter(F.col("rn") == 1)
    .select("automotor_tipo_descripcion", F.col("automotor_tipo_codigo").alias("codigo_tipo_nuevo"))
)

# 2.6. Cruzar y actualizar
df_silver = df_silver.join(df_maestro_tipo, on="automotor_tipo_descripcion", how="left")

df_silver = (
    df_silver.withColumn(
        "automotor_tipo_codigo", 
        F.coalesce(F.col("codigo_tipo_nuevo"), F.col("automotor_tipo_codigo"))
    )
    .drop("codigo_tipo_nuevo")
)

# ============================================================
# PASO 3: Asignación de IDs Sintéticos (Similar a Celda 13)
# ============================================================

# 3.1. Identificar los tipos que actualmente no tienen ID
df_tipo_sin_id = df_silver.filter(
    F.col("automotor_tipo_codigo").isNull() | 
    (F.lower(F.col("automotor_tipo_codigo")) == 'null') | 
    (F.col("automotor_tipo_codigo") == '')
)

# 3.2. Agrupar, contar y filtrar solo los que tienen más de 10 registros (DESTACADOS)
tipos_destacados = (
    df_tipo_sin_id.groupBy("automotor_tipo_descripcion")
    .agg(F.count("*").alias("conteo"))
    .filter(F.col("conteo") > 10)
)

# 3.3. Generar un ID único sintético para estos tipos destacados (comenzando en 8001)
window_spec_tipo_id = Window.orderBy(F.col("conteo").desc())

df_nuevos_ids_tipo = tipos_destacados.withColumn(
    "nuevo_codigo_tipo", 
    (F.row_number().over(window_spec_tipo_id) + 8000).cast("string")
).select("automotor_tipo_descripcion", "nuevo_codigo_tipo")

# 3.4. Cruzar con el DataFrame original
df_silver = df_silver.join(df_nuevos_ids_tipo, on="automotor_tipo_descripcion", how="left")

# 3.5. Actualizar la columna original solo donde aplique el nuevo código
df_silver = (
    df_silver.withColumn(
        "automotor_tipo_codigo", 
        F.coalesce(F.col("automotor_tipo_codigo"), F.col("nuevo_codigo_tipo"))
    )
    .drop("nuevo_codigo_tipo")
)

# 3.6. PARA TIPOS RAROS (<=10 registros): asignar IDs únicos desde 8999 en adelante
# 3.6.1. Identificar tipos que aún no tienen código
df_tipos_raros = df_silver.filter(
    F.col("automotor_tipo_codigo").isNull() | 
    (F.lower(F.col("automotor_tipo_codigo")) == 'null') | 
    (F.col("automotor_tipo_codigo") == '')
).select("automotor_tipo_descripcion").distinct()

# 3.6.2. Asignar IDs únicos empezando desde 8999
window_spec_tipos_raros = Window.orderBy("automotor_tipo_descripcion")

df_ids_tipos_raros = df_tipos_raros.withColumn(
    "codigo_tipo_raro",
    (F.row_number().over(window_spec_tipos_raros) + 8998).cast("string")
)

# 3.6.3. Cruzar y actualizar
df_silver = df_silver.join(df_ids_tipos_raros, on="automotor_tipo_descripcion", how="left")

df_silver = (
    df_silver.withColumn(
        "automotor_tipo_codigo",
        F.coalesce(F.col("automotor_tipo_codigo"), F.col("codigo_tipo_raro"))
    )
    .drop("codigo_tipo_raro")
)

print("✓ Limpieza de automotor_tipo_codigo y automotor_tipo_descripcion completada con IDs únicos")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# PASO 1: Normalización SQL por Frecuencia (Similar a Celda 10)
# ============================================================
df_silver.createOrReplaceTempView("df_silver_modelo_temp")

df_silver = spark.sql("""
  WITH modelo_normalizado AS (
    SELECT 
      automotor_modelo_codigo,
      automotor_modelo_descripcion,
      COUNT(*) as conteo,
      ROW_NUMBER() OVER (PARTITION BY automotor_modelo_codigo ORDER BY COUNT(*) DESC) as rank
    FROM df_silver_modelo_temp
    WHERE automotor_modelo_codigo IS NOT NULL
    GROUP BY automotor_modelo_codigo, automotor_modelo_descripcion
  )
  SELECT 
    t.tramite_tipo,
    t.tramite_fecha,
    t.fecha_inscripcion_inicial,
    t.registro_seccional_codigo,
    t.registro_seccional_descripcion,
    t.registro_seccional_provincia,
    t.automotor_origen,
    t.automotor_anio_modelo,
    t.automotor_tipo_codigo,
    t.automotor_tipo_descripcion,
    t.automotor_marca_codigo,
    t.automotor_marca_descripcion,
    t.automotor_modelo_codigo,
    COALESCE(mn.automotor_modelo_descripcion, t.automotor_modelo_descripcion) as automotor_modelo_descripcion,
    t.automotor_uso_codigo,
    t.automotor_uso_descripcion,
    t.titular_tipo_persona,
    t.titular_domicilio_localidad,
    t.titular_domicilio_provincia,
    t.titular_pais_nacimiento,
    t.titular_porcentaje_titularidad,
    t.titular_domicilio_provincia_id,
    t.tramite
  FROM df_silver_modelo_temp t
  LEFT JOIN modelo_normalizado mn 
    ON t.automotor_modelo_codigo = mn.automotor_modelo_codigo 
    AND mn.rank = 1
""")

# ============================================================
# PASO 2: Normalización PySpark Genérica (Similar a Celda 11)
# ============================================================

# 2.1. Limpieza base para TODAS las filas (caracteres raros y espacios extra)
df_silver = df_silver.withColumn(
    "modelo_temp",
    F.regexp_replace(F.trim(F.upper(F.col("automotor_modelo_descripcion"))), r'[^A-Z0-9 ./+\-]', '')
)
# Reemplazamos múltiples espacios internos por un solo espacio
df_silver = df_silver.withColumn(
    "modelo_temp",
    F.regexp_replace(F.col("modelo_temp"), r'\s+', ' ')
)

# 2.2. Para modelos, aplicamos la versión limpia directamente
# (no usamos diccionario extenso porque hay miles de modelos diferentes)
df_silver = df_silver.withColumn(
    "automotor_modelo_descripcion",
    F.col("modelo_temp")
).drop("modelo_temp")

# 2.3. Calcular y asignar el ID Maestro para TODOS los modelos normalizados
df_codigos_modelo_validos = df_silver.filter(
    F.col("automotor_modelo_codigo").isNotNull() & 
    (F.lower(F.col("automotor_modelo_codigo")) != 'null') & 
    (F.col("automotor_modelo_codigo") != '')
).groupBy("automotor_modelo_descripcion", "automotor_modelo_codigo").agg(F.count("*").alias("conteo"))

window_spec_modelo = Window.partitionBy("automotor_modelo_descripcion").orderBy(F.col("conteo").desc())

df_maestro_modelo = (
    df_codigos_modelo_validos.withColumn("rn", F.row_number().over(window_spec_modelo))
    .filter(F.col("rn") == 1)
    .select("automotor_modelo_descripcion", F.col("automotor_modelo_codigo").alias("codigo_modelo_nuevo"))
)

# 2.4. Cruzar y actualizar
df_silver = df_silver.join(df_maestro_modelo, on="automotor_modelo_descripcion", how="left")

df_silver = (
    df_silver.withColumn(
        "automotor_modelo_codigo", 
        F.coalesce(F.col("codigo_modelo_nuevo"), F.col("automotor_modelo_codigo"))
    )
    .drop("codigo_modelo_nuevo")
)

# ============================================================
# PASO 3: Asignación de IDs Sintéticos (Similar a Celda 13)
# ============================================================

# 3.1. Identificar los modelos que actualmente no tienen ID
df_modelo_sin_id = df_silver.filter(
    F.col("automotor_modelo_codigo").isNull() | 
    (F.lower(F.col("automotor_modelo_codigo")) == 'null') | 
    (F.col("automotor_modelo_codigo") == '')
)

# 3.2. Agrupar, contar y filtrar solo los que tienen más de 10 registros (DESTACADOS)
modelos_destacados = (
    df_modelo_sin_id.groupBy("automotor_modelo_descripcion")
    .agg(F.count("*").alias("conteo"))
    .filter(F.col("conteo") > 10)
)

# 3.3. Generar un ID único sintético para estos modelos destacados (comenzando en 7001)
window_spec_modelo_id = Window.orderBy(F.col("conteo").desc())

df_nuevos_ids_modelo = modelos_destacados.withColumn(
    "nuevo_codigo_modelo", 
    (F.row_number().over(window_spec_modelo_id) + 7000).cast("string")
).select("automotor_modelo_descripcion", "nuevo_codigo_modelo")

# 3.4. Cruzar con el DataFrame original
df_silver = df_silver.join(df_nuevos_ids_modelo, on="automotor_modelo_descripcion", how="left")

# 3.5. Actualizar la columna original solo donde aplique el nuevo código
df_silver = (
    df_silver.withColumn(
        "automotor_modelo_codigo", 
        F.coalesce(F.col("automotor_modelo_codigo"), F.col("nuevo_codigo_modelo"))
    )
    .drop("nuevo_codigo_modelo")
)

# 3.6. PARA MODELOS RAROS (<=10 registros): asignar IDs únicos desde 7999 en adelante
# 3.6.1. Identificar modelos que aún no tienen código
df_modelos_raros = df_silver.filter(
    F.col("automotor_modelo_codigo").isNull() | 
    (F.lower(F.col("automotor_modelo_codigo")) == 'null') | 
    (F.col("automotor_modelo_codigo") == '')
).select("automotor_modelo_descripcion").distinct()

# 3.6.2. Asignar IDs únicos empezando desde 7999
window_spec_modelos_raros = Window.orderBy("automotor_modelo_descripcion")

df_ids_modelos_raros = df_modelos_raros.withColumn(
    "codigo_modelo_raro",
    (F.row_number().over(window_spec_modelos_raros) + 7998).cast("string")
)

# 3.6.3. Cruzar y actualizar
df_silver = df_silver.join(df_ids_modelos_raros, on="automotor_modelo_descripcion", how="left")

df_silver = (
    df_silver.withColumn(
        "automotor_modelo_codigo",
        F.coalesce(F.col("automotor_modelo_codigo"), F.col("codigo_modelo_raro"))
    )
    .drop("codigo_modelo_raro")
)

print("✓ Limpieza de automotor_modelo_codigo y automotor_modelo_descripcion completada con IDs únicos")

In [0]:
df_silver = df_silver.withColumn(
    "id_tramite",
    F.concat_ws(
        "-",
        F.col("registro_seccional_codigo"),
        F.col("automotor_marca_codigo"),
        F.col("automotor_modelo_codigo"),
        F.col("automotor_tipo_codigo"),
        F.date_format(F.col("tramite_fecha"), "yyyy_MM_dd")
    )
)

In [0]:
#Eliminamos duplicados
df_silver = df_silver.dropDuplicates(["id_tramite"])

In [0]:
df_silver = df_silver.createOrReplaceTempView("df_silver_temp_cast")


df_silver = spark.sql("""
SELECT
    TRY_CAST(tramite_fecha AS DATE) AS tramite_fecha,
    TRY_CAST(fecha_inscripcion_inicial AS DATE) AS fecha_inscripcion_inicial,
    CAST(registro_seccional_codigo AS STRING) AS registro_seccional_codigo,
    CAST(registro_seccional_descripcion AS STRING) AS registro_seccional_descripcion,
    CAST(registro_seccional_provincia AS STRING) AS registro_seccional_provincia,
    CAST(automotor_origen AS STRING) AS automotor_origen,
    CAST(automotor_anio_modelo AS STRING) AS automotor_anio_modelo,
    CAST(automotor_tipo_codigo AS STRING) AS automotor_tipo_codigo,
    CAST(automotor_tipo_descripcion AS STRING) AS automotor_tipo_descripcion,
    CAST(automotor_marca_codigo AS STRING) AS automotor_marca_codigo,
    CAST(automotor_marca_descripcion AS STRING) AS automotor_marca_descripcion,
    CAST(automotor_modelo_codigo AS STRING) AS automotor_modelo_codigo,
    CAST(automotor_modelo_descripcion AS STRING) AS automotor_modelo_descripcion,
    CAST(automotor_uso_codigo AS STRING) AS automotor_uso_codigo,
    CAST(automotor_uso_descripcion AS STRING) AS automotor_uso_descripcion,
    CAST(titular_tipo_persona AS STRING) AS titular_tipo_persona,
    CAST(titular_domicilio_localidad AS STRING) AS titular_domicilio_localidad,
    CAST(titular_domicilio_provincia AS STRING) AS titular_domicilio_provincia,
    CAST(titular_pais_nacimiento AS STRING) AS titular_pais_nacimiento,
    CAST(titular_porcentaje_titularidad AS STRING) AS titular_porcentaje_titularidad,
    CAST(titular_domicilio_provincia_id AS STRING) AS titular_domicilio_provincia_id,
    CAST(tramite AS STRING) AS tramite,
    CAST(id_tramite AS STRING) AS id_tramite
FROM df_silver_temp_cast
""")

#Guardamos df_silver en delta

In [0]:
# Creamos el schema para Silver si no existe
#spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.tp_dnrpa_silver")
print("✓ Schema tp_dnrpa_silver creado")

In [0]:
# ==============================================================================
# PASO 6.5: AUDITORÍA Y DATA QUALITY FINAL (Antes de materializar)
# ==============================================================================

columnas = [c for c in df_silver.columns if c != "silver_ingestion_timestamp"]

# 1. Tratamiento de strings vacíos y 2. Timestamp 
df_silver = (
    df_silver.select([
        F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c)).alias(c) 
        for c in columnas
    ])
    .withColumn("silver_ingestion_timestamp", F.current_timestamp())
)

# ==============================================================================
# PASO 7: GUARDADO (Materialización Delta Idempotente)
# ==============================================================================

# 3. Escritura final usando tu única variable df_silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")                 
    .option("overwriteSchema", "true") 
    .saveAsTable("workspace.tp_dnrpa_silver.silver_transferencias")
)